# Import Dependencies

In [1]:
import torch
from urllib.request import urlopen
from PIL import Image
from open_clip import create_model_from_pretrained, get_tokenizer

# Load BiomedCLIP

In [16]:

# Load the model and config files from the Hugging Face Hub
model, preprocess = create_model_from_pretrained('hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224')
tokenizer = get_tokenizer('hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224')

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(f'Using device: {device}')
model.to(device)
model.eval()


Using device: cuda


CustomTextCLIP(
  (visual): TimmModel(
    (trunk): VisionTransformer(
      (patch_embed): PatchEmbed(
        (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
        (norm): Identity()
      )
      (pos_drop): Dropout(p=0.0, inplace=False)
      (patch_drop): Identity()
      (norm_pre): Identity()
      (blocks): Sequential(
        (0): Block(
          (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (attn): Attention(
            (qkv): Linear(in_features=768, out_features=2304, bias=True)
            (q_norm): Identity()
            (k_norm): Identity()
            (attn_drop): Dropout(p=0.0, inplace=False)
            (proj): Linear(in_features=768, out_features=768, bias=True)
            (proj_drop): Dropout(p=0.0, inplace=False)
          )
          (ls1): Identity()
          (drop_path1): Identity()
          (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (mlp): Mlp(
            (fc1): Linear(in_features=768

# Set up labels and list of imaged from DIBaS_Dataset

In [17]:
# Zero-shot image classification
template = 'this is a photo of '
labels = [
"Acinetobacter.baumanii",
"Actinomyces.israeli",
"Bacteroides.fragilis",
"Bifidobacterium.spp",
"Candida.albicans",
"Clostridium.perfringens",
"Enterococcus.faecium",
"Enterococcus.faecalis",
"Escherichia.coli",
"Fusobacterium",
"Lactobacillus.casei",
"Lactobacillus.crispatus",
"Lactobacillus.delbrueckii",
"Lactobacillus.gasseri",
"Lactobacillus.jehnsenii",
"Lactobacillus.johnsonii",
"Lactobacillus.paracasei",
"Lactobacillus.plantarum",
"Lactobacillus.reuteri",
"Lactobacillus.rhamnosus",
"Lactobacillus.salivarius",
"Listeria.monocytogenes",
"Micrococcus.spp",
"Neisseria.gonorrhoeae",
"Porfyromonas.gingivalis",
"Propionibacterium.acnes",
"Proteus",
"Pseudomonas.aeruginosa",
"Staphylococcus.aureus",
"Staphylococcus.epidermidis",
"Staphylococcus.saprophiticus",
"Streptococcus.agalactiae",
"Veionella"]

test_imgs = [
    '../data/DIBaS_Dataset/Acinetobacter.baumanii/Acinetobacter.baumanii_0001.tif',
    '../data/DIBaS_Dataset/Actinomyces.israeli/Actinomyces.israeli_0001.tif',
    '../data/DIBaS_Dataset/Veionella/Veionella_0001.tif',
    '../data/DIBaS_Dataset/Streptococcus.agalactiae/Streptococcus.agalactiae_0001.tif'
]


# Run inference to classify images

In [ ]:

context_length = 256

images = torch.stack([preprocess(Image.open(img)) for img in test_imgs]).to(device)
texts = tokenizer([template + l for l in labels], context_length=context_length).to(device)
with torch.no_grad():
    image_features, text_features, logit_scale = model(images, texts)

    logits = (logit_scale * image_features @ text_features.t()).detach().softmax(dim=-1)
    sorted_indices = torch.argsort(logits, dim=-1, descending=True)

    logits = logits.cpu().numpy()
    sorted_indices = sorted_indices.cpu().numpy()

top_k = -1

for i, img in enumerate(test_imgs):
    pred = labels[sorted_indices[i][0]]

    top_k = len(labels) if top_k == -1 else top_k
    print(img.split('/')[-1] + ':')
    for j in range(top_k):
        jth_index = sorted_indices[i][j]
        print(f'{labels[jth_index]}: {logits[i][jth_index]}')
    print('\n')

Acinetobacter.baumanii_0001.tif:
Listeria.monocytogenes: 0.2861093282699585
Lactobacillus.johnsonii: 0.1759343296289444
Lactobacillus.casei: 0.15312692523002625
Fusobacterium: 0.12113892287015915
Lactobacillus.delbrueckii: 0.0772138386964798
Lactobacillus.plantarum: 0.04591377079486847
Lactobacillus.rhamnosus: 0.02897922322154045
Lactobacillus.crispatus: 0.027897659689188004
Escherichia.coli: 0.020612703636288643
Candida.albicans: 0.0136687271296978
Streptococcus.agalactiae: 0.010990716516971588
Lactobacillus.paracasei: 0.01037828903645277
Clostridium.perfringens: 0.007313133217394352
Lactobacillus.jehnsenii: 0.007310009095817804
Lactobacillus.gasseri: 0.005712576676160097
Lactobacillus.reuteri: 0.0013838285813108087
Staphylococcus.aureus: 0.0011829426512122154
Enterococcus.faecalis: 0.001050887512974441
Neisseria.gonorrhoeae: 0.0008794672903604805
Lactobacillus.salivarius: 0.0007430192781612277
Micrococcus.spp: 0.0007392280967906117
Acinetobacter.baumanii: 0.0006754772621206939
Staphy

# YEP - No Bueno